# OA Potential Fit with Energy-Tied Alpha Weighting

Builds a Fourier-Morse potential for the PA/OA interaction where each
per-orientation Morse ``alpha`` is fitted with an **energy-tied weight**

$$w(r) = \sqrt{\max\big(E_{\min} - E(r) - E(r_e),\ \epsilon\big)},\qquad \epsilon = 10^{-4}$$

where $E_{\min}$ is the global minimum energy of the data, $E(r)$ the energy
at radial distance $r$, and $E(r_e)$ the per-orientation minimum energy. The
weight grows toward the deep minimum and is suppressed where $E(r)$ rises
(the steep repulsive side).

Harmonic ceilings are OA $(20, 1)$, matching the equal-weight baseline of
``plot_oa_error.py``. The fitted surface is exported to a CSV for
``plot_oa_error.py --fit-input``.

In [ ]:
from pathlib import Path

from chimorse.config import load_molecule_info
from chimorse.datasets import ensure_reference_data
from chimorse.dataio import load_data
from chimorse.fitting import generate_fourier_morse_data, make_weight_func

In [ ]:
molecule_name = 'PA'
interaction   = 'OA'
zero_zeta     = True
alpha_fit     = True

# Harmonic ceilings — OA (20, 1) to match the equal-weight baseline.
harmonic_ceils = {'EP': (8, 1), 'EA': (8, 1), 'OP': (20, 1), 'OA': (20, 1)}

# Energy-tied weight for the alpha fit (eps defaults to 1e-4).
weight_func = make_weight_func('energy', eps=1e-4)

data_root = Path('../data')
data_dir = ensure_reference_data(molecule_name, data_root=data_root)
molecule = load_molecule_info(molecule_name, metadata_path=data_dir / 'metadata.json')
df = load_data(molecule, interaction, zero_zeta=zero_zeta)
print(f'{molecule.name} {interaction}: {len(df)} rows')

In [ ]:
df_model = generate_fourier_morse_data(
    df, molecule, interaction, harmonic_ceils,
    alpha_fit=alpha_fit, weight_func=weight_func,
    print_errors=True, near_eq_delta_r=.5,
)
print('model rows:', len(df_model))

In [ ]:
out_csv = data_dir / f'df_model_{interaction}_w_energy.csv'
df_model.to_csv(out_csv, index=False)
print('saved fit:', out_csv)

### Next step

Run ``plot_oa_error.py --fit-input <out_csv> --tag w_energy``
to generate the corresponding error plots.